CNN 

In [ ]:
import os
import numpy as np
import pandas as pd
from PIL import Image

from sklearn.model_selection import train_test_split

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv2D,
    MaxPooling2D,
    Flatten,
    Dense,
    Dropout
)


df = pd.read_csv(
    r"C:\Users\alijo\OneDrive\Desktop\iivp-2026-challenge\train_augmented.csv"
)

train_dir = r"C:\Users\alijo\OneDrive\Desktop\iivp-2026-challenge\train_augmented"

X = []
y = []

for _, row in df.iterrows():

    img_path = os.path.join(
        train_dir,
        str(row["Category"]),
        str(row["Id"]) + ".png"
    )

    img = Image.open(img_path).convert("L")

    # normalize
    img = np.array(img) / 255.0

    X.append(img)
    y.append(row["Category"])



X = np.array(X)
y = np.array(y)

print("Dataset shape:", X.shape)

X = X.reshape(-1, 32, 32, 1)

print("CNN shape:", X.shape)


X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

model = Sequential([

    Conv2D(
        32,
        (3,3),
        activation='relu',
        input_shape=(32,32,1)
    ),

    MaxPooling2D((2,2)),

    Conv2D(
        64,
        (3,3),
        activation='relu'
    ),

    MaxPooling2D((2,2)),

    Flatten(),

    Dense(128, activation='relu'),

    Dropout(0.5),

    Dense(10, activation='softmax')
])


model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)


history = model.fit(
    X_train,
    y_train,
    epochs=10,
    batch_size=32,
    validation_data=(X_val, y_val)
)

val_loss, val_acc = model.evaluate(X_val, y_val)

print("Validation Accuracy:", val_acc)



X_test = []
names = []

test_dir = r"C:\Users\alijo\OneDrive\Desktop\iivp-2026-challenge\test\test"

for file in sorted(os.listdir(test_dir)):

    img_path = os.path.join(test_dir, file)

    img = Image.open(img_path).convert("L")

    img = np.array(img) / 255.0

    X_test.append(img)
    names.append(file)

X_test = np.array(X_test)

# reshape for CNN
X_test = X_test.reshape(-1, 32, 32, 1)
preds = model.predict(X_test)

pred_labels = np.argmax(preds, axis=1)

submission = pd.DataFrame({
    "Id": names,
    "Category": pred_labels
})

submission["Id"] = submission["Id"].str.replace(
    ".png",
    "",
    regex=False
)

submission.to_csv("submission_cnn.csv", index=False)

print("submission_cnn.csv saved")

Dataset shape: (34000, 32, 32)
CNN shape: (34000, 32, 32, 1)


c:\Users\alijo\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10
850/850 ━━━━━━━━━━━━━━━━━━━━ 9s 8ms/step - accuracy: 0.8729 - loss: 0.3956 - val_accuracy: 0.9701 - val_loss: 0.1015
Epoch 2/10
850/850 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - accuracy: 0.9569 - loss: 0.1362 - val_accuracy: 0.9788 - val_loss: 0.0671
Epoch 3/10
850/850 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - accuracy: 0.9724 - loss: 0.0913 - val_accuracy: 0.9829 - val_loss: 0.0540
Epoch 4/10
850/850 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - accuracy: 0.9789 - loss: 0.0663 - val_accuracy: 0.9857 - val_loss: 0.0436
Epoch 5/10
850/850 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - accuracy: 0.9833 - loss: 0.0520 - val_accuracy: 0.9865 - val_loss: 0.0398
Epoch 6/10
850/850 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - accuracy: 0.9864 - loss: 0.0421 - val_accuracy: 0.9893 - val_loss: 0.0348
Epoch 7/10
850/850 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - accuracy: 0.9886 - loss: 0.0353 - val_accuracy: 0.9910 - val_loss: 0.0312
Epoch 8/10
850/850 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - accuracy: 0.9903 - loss: 0.0304 - val_accuracy: 0.

CNN With BatchNorm

In [ ]:
import os
import numpy as np
import pandas as pd
from PIL import Image

from sklearn.model_selection import train_test_split

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    BatchNormalization
)

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau
)


df = pd.read_csv(
    r"C:\Users\alijo\OneDrive\Desktop\iivp-2026-challenge\train_augmented.csv"
)



train_dir = r"C:\Users\alijo\OneDrive\Desktop\iivp-2026-challenge\train_V4"

X = []
y = []


for _, row in df.iterrows():

    img_path = os.path.join(
        train_dir,
        str(row["Category"]),
        str(row["Id"]) + ".png"
    )

    img = Image.open(img_path).convert("L")

    # normalize pixels
    img = np.array(img) / 255.0

    X.append(img)
    y.append(row["Category"])



X = np.array(X)
y = np.array(y)

print("Dataset shape:", X.shape)


X = X.reshape(-1, 32, 32, 1)

print("CNN shape:", X.shape)


X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)


model = Sequential([


    Conv2D(
        32,
        (3,3),
        activation='relu',
        input_shape=(32,32,1)
    ),

    BatchNormalization(),

    MaxPooling2D((2,2)),

 
    Conv2D(
        64,
        (3,3),
        activation='relu'
    ),

    BatchNormalization(),

    MaxPooling2D((2,2)),



    Conv2D(
        128,
        (3,3),
        activation='relu'
    ),

    BatchNormalization(),


    Flatten(),

    Dense(256, activation='relu'),

    Dropout(0.5),

    Dense(10, activation='softmax')
])



model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)



early_stop = EarlyStopping(
    monitor='val_accuracy',
    patience=5,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=2,
    min_lr=1e-6
)

history = model.fit(
    X_train,
    y_train,
    epochs=30,
    batch_size=32,
    validation_data=(X_val, y_val),
    callbacks=[early_stop, reduce_lr]
)


val_loss, val_acc = model.evaluate(X_val, y_val)

print("Validation Accuracy:", val_acc)


X_test = []
names = []

test_dir = r"C:\Users\alijo\OneDrive\Desktop\iivp-2026-challenge\test\test"

for file in sorted(os.listdir(test_dir)):

    img_path = os.path.join(test_dir, file)

    img = Image.open(img_path).convert("L")

    img = np.array(img) / 255.0

    X_test.append(img)
    names.append(file)



X_test = np.array(X_test)

X_test = X_test.reshape(-1, 32, 32, 1)

preds = model.predict(X_test)

pred_labels = np.argmax(preds, axis=1)

submission = pd.DataFrame({
    "Id": names,
    "Category": pred_labels
})

# remove .png extension
submission["Id"] = submission["Id"].str.replace(
    ".png",
    "",
    regex=False
)


submission.to_csv(
    "submission_batchnorm.csv",
    index=False
)

print("submission_batchnorm.csv saved")

Dataset shape: (34000, 32, 32)
CNN shape: (34000, 32, 32, 1)


c:\Users\alijo\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/30
850/850 ━━━━━━━━━━━━━━━━━━━━ 16s 16ms/step - accuracy: 0.9226 - loss: 0.2664 - val_accuracy: 0.9815 - val_loss: 0.0640 - learning_rate: 0.0010
Epoch 2/30
850/850 ━━━━━━━━━━━━━━━━━━━━ 14s 16ms/step - accuracy: 0.9778 - loss: 0.0791 - val_accuracy: 0.9862 - val_loss: 0.0538 - learning_rate: 0.0010
Epoch 3/30
850/850 ━━━━━━━━━━━━━━━━━━━━ 13s 16ms/step - accuracy: 0.9844 - loss: 0.0570 - val_accuracy: 0.9847 - val_loss: 0.0650 - learning_rate: 0.0010
Epoch 4/30
850/850 ━━━━━━━━━━━━━━━━━━━━ 14s 16ms/step - accuracy: 0.9861 - loss: 0.0520 - val_accuracy: 0.9828 - val_loss: 0.0709 - learning_rate: 0.0010
Epoch 5/30
850/850 ━━━━━━━━━━━━━━━━━━━━ 14s 16ms/step - accuracy: 0.9932 - loss: 0.0216 - val_accuracy: 0.9946 - val_loss: 0.0276 - learning_rate: 5.0000e-04
Epoch 6/30
850/850 ━━━━━━━━━━━━━━━━━━━━ 14s 16ms/step - accuracy: 0.9961 - loss: 0.0124 - val_accuracy: 0.9946 - val_loss: 0.0237 - learning_rate: 5.0000e-04
Epoch 7/30
850/850 ━━━━━━━━━━━━━━━━━━━━ 13s 16ms/step - accuracy: 0.

MNIST Style CNN

In [ ]:

from tensorflow.keras.preprocessing.image import ImageDataGenerator


df = pd.read_csv(
    r"C:\Users\alijo\OneDrive\Desktop\iivp-2026-challenge\train_V4\train_V3_labels.csv"
)


train_dir = r"C:\Users\alijo\OneDrive\Desktop\iivp-2026-challenge\train_V4"

X = []
y = []

for _, row in df.iterrows():

    img_path = os.path.join(
        train_dir,
        str(row["Category"]),
        str(row["Id"]) + ".png"
    )

    img = Image.open(img_path).convert("L")

    # normalize
    img = np.array(img) / 255.0

    X.append(img)
    y.append(row["Category"])


X = np.array(X)
y = np.array(y)

print("Dataset shape:", X.shape)


X = X.reshape(-1, 32, 32, 1)

print("CNN shape:", X.shape)


X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

datagen = ImageDataGenerator(

    rotation_range=6,

    width_shift_range=0.05,

    height_shift_range=0.05,

    zoom_range=0.05
)



model = Sequential([

    Conv2D(
        32,
        (3,3),
        activation='relu',
        padding='same',
        input_shape=(32,32,1)
    ),

    BatchNormalization(),

    Conv2D(
        32,
        (3,3),
        activation='relu',
        padding='same'
    ),

    BatchNormalization(),

    MaxPooling2D((2,2)),

    Dropout(0.25),


    Conv2D(
        64,
        (3,3),
        activation='relu',
        padding='same'
    ),

    BatchNormalization(),

    Conv2D(
        64,
        (3,3),
        activation='relu',
        padding='same'
    ),

    BatchNormalization(),

    MaxPooling2D((2,2)),

    Dropout(0.25),


    Conv2D(
        128,
        (3,3),
        activation='relu',
        padding='same'
    ),

    BatchNormalization(),

    Conv2D(
        128,
        (3,3),
        activation='relu',
        padding='same'
    ),

    BatchNormalization(),

    MaxPooling2D((2,2)),

    Dropout(0.25),

    Flatten(),

    Dense(256, activation='relu'),

    BatchNormalization(),

    Dropout(0.5),

    Dense(10, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)


early_stop = EarlyStopping(
    monitor='val_accuracy',
    patience=5,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=2,
    min_lr=1e-6
)


history = model.fit(
    datagen.flow(X_train, y_train, batch_size=32),
    epochs=40,
    validation_data=(X_val, y_val),
    callbacks=[early_stop, reduce_lr]
)


val_loss, val_acc = model.evaluate(X_val, y_val)

print("Validation Accuracy:", val_acc)


X_test = []
names = []

test_dir = r"C:\Users\alijo\OneDrive\Desktop\iivp-2026-challenge\test\test"

for file in sorted(os.listdir(test_dir)):

    img_path = os.path.join(test_dir, file)

    img = Image.open(img_path).convert("L")

    img = np.array(img) / 255.0

    X_test.append(img)
    names.append(file)


X_test = np.array(X_test)

X_test = X_test.reshape(-1, 32, 32, 1)


preds = model.predict(X_test)

pred_labels = np.argmax(preds, axis=1)


submission = pd.DataFrame({
    "Id": names,
    "Category": pred_labels
})

submission["Id"] = submission["Id"].str.replace(
    ".png",
    "",
    regex=False
)

submission.to_csv(
    "submission_supercnn_V4.csv",
    index=False
)

print("submission_supercnn.csv saved")

Dataset shape: (34000, 32, 32)
CNN shape: (34000, 32, 32, 1)


c:\Users\alijo\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/40


c:\Users\alijo\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


850/850 ━━━━━━━━━━━━━━━━━━━━ 46s 48ms/step - accuracy: 0.9110 - loss: 0.2800 - val_accuracy: 0.9854 - val_loss: 0.0478 - learning_rate: 0.0010
Epoch 2/40
850/850 ━━━━━━━━━━━━━━━━━━━━ 37s 43ms/step - accuracy: 0.9774 - loss: 0.0693 - val_accuracy: 0.9926 - val_loss: 0.0290 - learning_rate: 0.0010
Epoch 3/40
850/850 ━━━━━━━━━━━━━━━━━━━━ 37s 43ms/step - accuracy: 0.9842 - loss: 0.0478 - val_accuracy: 0.9934 - val_loss: 0.0229 - learning_rate: 0.0010
Epoch 4/40
850/850 ━━━━━━━━━━━━━━━━━━━━ 36s 43ms/step - accuracy: 0.9875 - loss: 0.0387 - val_accuracy: 0.9941 - val_loss: 0.0173 - learning_rate: 0.0010
Epoch 5/40
850/850 ━━━━━━━━━━━━━━━━━━━━ 42s 43ms/step - accuracy: 0.9894 - loss: 0.0319 - val_accuracy: 0.9940 - val_loss: 0.0175 - learning_rate: 0.0010
Epoch 6/40
850/850 ━━━━━━━━━━━━━━━━━━━━ 37s 43ms/step - accuracy: 0.9903 - loss: 0.0326 - val_accuracy: 0.9943 - val_loss: 0.0186 - learning_rate: 0.0010
Epoch 7/40
850/850 ━━━━━━━━━━━━━━━━━━━━ 37s 43ms/step - accuracy: 0.9944 - loss: 0.0167

___________________________________________________________________________________________________________________________
___________________________________________________________________________________________________________________________
___________________________________________________________________________________________________________________________
___________________________________________________________________________________________________________________________

Shared setup for resnet and efficientNet


In [ ]:
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
from torchvision.datasets import ImageFolder
import torchvision.transforms as T
from torchvision import models
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR
import pandas as pd


train_root = r"C:\Users\alijo\OneDrive\Desktop\iivp-2026-challenge\train\train"
test_root  = r"C:\Users\alijo\OneDrive\Desktop\iivp-2026-challenge\test\test"


train_transform = T.Compose([
    T.Grayscale(num_output_channels=3),
    T.Resize((64, 64)),
    T.RandomRotation(10),
    T.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.95, 1.05), shear=5),
    T.GaussianBlur(kernel_size=3),
    T.ToTensor(),
    T.Normalize([0.5]*3, [0.5]*3)
])

test_transform = T.Compose([
    T.Grayscale(num_output_channels=3),
    T.Resize((64, 64)),
    T.ToTensor(),
    T.Normalize([0.5]*3, [0.5]*3)
])


full_dataset = ImageFolder(train_root, transform=train_transform)
num_classes = len(full_dataset.classes)
print(f"Classes: {full_dataset.classes}")

val_size = int(0.2 * len(full_dataset))
train_size = len(full_dataset) - val_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False, num_workers=0)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Classes: ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9']
Using device: cpu


Train efficientNet


In [ ]:
efficientnet = models.efficientnet_b0(weights="IMAGENET1K_V1")
efficientnet.classifier[1] = nn.Linear(efficientnet.classifier[1].in_features, num_classes)
efficientnet = efficientnet.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = Adam(efficientnet.parameters(), lr=1e-3)
scheduler = CosineAnnealingLR(optimizer, T_max=20)

best_acc = 0
for epoch in range(20):
    efficientnet.train()
    total_loss, correct, total = 0, 0, 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = efficientnet(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)

    efficientnet.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = efficientnet(imgs)
            val_correct += (outputs.argmax(1) == labels).sum().item()
            val_total += labels.size(0)

    train_acc = correct / total
    val_acc = val_correct / val_total
    scheduler.step()
    print(f"[EfficientNet] Epoch {epoch+1}/20 | Loss: {total_loss/len(train_loader):.4f} | Train: {train_acc:.4f} | Val: {val_acc:.4f}")

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(efficientnet.state_dict(), "efficientnet_best.pth")
        print(f"  ✅ Saved (val_acc={val_acc:.4f})")

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to C:\Users\alijo/.cache\torch\hub\checkpoints\efficientnet_b0_rwightman-7f5810bc.pth
100%|██████████| 20.5M/20.5M [00:02<00:00, 10.3MB/s]


[EfficientNet] Epoch 1/20 | Loss: 0.2764 | Train: 0.9192 | Val: 0.9921
  ✅ Saved (val_acc=0.9921)
[EfficientNet] Epoch 2/20 | Loss: 0.0742 | Train: 0.9795 | Val: 0.9947
  ✅ Saved (val_acc=0.9947)
[EfficientNet] Epoch 3/20 | Loss: 0.0423 | Train: 0.9865 | Val: 0.9935
[EfficientNet] Epoch 4/20 | Loss: 0.0329 | Train: 0.9901 | Val: 0.9959
  ✅ Saved (val_acc=0.9959)
[EfficientNet] Epoch 5/20 | Loss: 0.0334 | Train: 0.9920 | Val: 0.9979
  ✅ Saved (val_acc=0.9979)
[EfficientNet] Epoch 6/20 | Loss: 0.0239 | Train: 0.9931 | Val: 0.9944
[EfficientNet] Epoch 7/20 | Loss: 0.0207 | Train: 0.9930 | Val: 0.9921
[EfficientNet] Epoch 8/20 | Loss: 0.0169 | Train: 0.9960 | Val: 0.9988
  ✅ Saved (val_acc=0.9988)
[EfficientNet] Epoch 9/20 | Loss: 0.0160 | Train: 0.9957 | Val: 0.9965
[EfficientNet] Epoch 10/20 | Loss: 0.0092 | Train: 0.9972 | Val: 0.9994
  ✅ Saved (val_acc=0.9994)
[EfficientNet] Epoch 11/20 | Loss: 0.0069 | Train: 0.9979 | Val: 0.9982
[EfficientNet] Epoch 12/20 | Loss: 0.0081 | Train: 0.99

Train Resnet

In [ ]:
resnet = models.resnet18(weights="IMAGENET1K_V1")
resnet.fc = nn.Linear(resnet.fc.in_features, num_classes)
resnet = resnet.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = Adam(resnet.parameters(), lr=1e-3)
scheduler = CosineAnnealingLR(optimizer, T_max=20)

best_acc = 0
for epoch in range(20):
    resnet.train()
    total_loss, correct, total = 0, 0, 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = resnet(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)

    resnet.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = resnet(imgs)
            val_correct += (outputs.argmax(1) == labels).sum().item()
            val_total += labels.size(0)

    train_acc = correct / total
    val_acc = val_correct / val_total
    scheduler.step()
    print(f"[ResNet18] Epoch {epoch+1}/20 | Loss: {total_loss/len(train_loader):.4f} | Train: {train_acc:.4f} | Val: {val_acc:.4f}")

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(resnet.state_dict(), "resnet_best.pth")
        print(f"  ✅ Saved (val_acc={val_acc:.4f})")

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\alijo/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth
100%|██████████| 44.7M/44.7M [00:04<00:00, 10.8MB/s]


[ResNet18] Epoch 1/20 | Loss: 0.1773 | Train: 0.9496 | Val: 0.9844
  ✅ Saved (val_acc=0.9844)
[ResNet18] Epoch 2/20 | Loss: 0.0808 | Train: 0.9787 | Val: 0.9956
  ✅ Saved (val_acc=0.9956)
[ResNet18] Epoch 3/20 | Loss: 0.0568 | Train: 0.9860 | Val: 0.9950
[ResNet18] Epoch 4/20 | Loss: 0.0326 | Train: 0.9908 | Val: 0.9806
[ResNet18] Epoch 5/20 | Loss: 0.0494 | Train: 0.9871 | Val: 0.9974
  ✅ Saved (val_acc=0.9974)
[ResNet18] Epoch 6/20 | Loss: 0.0234 | Train: 0.9934 | Val: 0.9879
[ResNet18] Epoch 7/20 | Loss: 0.0241 | Train: 0.9935 | Val: 0.9962
[ResNet18] Epoch 8/20 | Loss: 0.0180 | Train: 0.9951 | Val: 0.9985
  ✅ Saved (val_acc=0.9985)
[ResNet18] Epoch 9/20 | Loss: 0.0158 | Train: 0.9954 | Val: 0.9982
[ResNet18] Epoch 10/20 | Loss: 0.0160 | Train: 0.9961 | Val: 0.9418
[ResNet18] Epoch 11/20 | Loss: 0.0177 | Train: 0.9957 | Val: 0.9965
[ResNet18] Epoch 12/20 | Loss: 0.0140 | Train: 0.9963 | Val: 0.9982
[ResNet18] Epoch 13/20 | Loss: 0.0062 | Train: 0.9979 | Val: 0.9994
  ✅ Saved (val_ac

Ensemble

In [ ]:
from torch.utils.data import Dataset
from PIL import Image

class TestDataset(Dataset):
    def __init__(self, root, transform=None):
        self.root = root
        self.transform = transform
        self.images = sorted([f for f in os.listdir(root) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = os.path.join(self.root, self.images[idx])
        img = Image.open(img_path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, self.images[idx]

test_dataset = TestDataset(test_root, transform=test_transform)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=0)

efficientnet = models.efficientnet_b0(weights=None)
efficientnet.classifier[1] = nn.Linear(efficientnet.classifier[1].in_features, num_classes)
efficientnet.load_state_dict(torch.load("efficientnet_best.pth"))
efficientnet = efficientnet.to(device).eval()

resnet = models.resnet18(weights=None)
resnet.fc = nn.Linear(resnet.fc.in_features, num_classes)
resnet.load_state_dict(torch.load("resnet_best.pth"))
resnet = resnet.to(device).eval()


all_preds = []
all_ids   = []

with torch.no_grad():
    for imgs, filenames in test_loader:
        imgs = imgs.to(device)
        probs_eff = torch.softmax(efficientnet(imgs), dim=1)
        probs_res = torch.softmax(resnet(imgs), dim=1)
        avg_probs = (probs_eff + probs_res) / 2
        preds = avg_probs.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_ids.extend(filenames)


image_ids = [os.path.splitext(f)[0] for f in all_ids]
idx_to_class = {v: k for k, v in full_dataset.class_to_idx.items()}
categories = [idx_to_class[p] for p in all_preds]

df = pd.DataFrame({"Id": image_ids, "Category": categories})
df.to_csv("submission_ensemble.csv", index=False)
print(f"✅ Saved submission_ensemble.csv with {len(df)} rows")

✅ Saved submission_ensemble.csv with 3000 rows
